# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Климишин Данило'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-31'
НОМЕР        = 8       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 8   ·   Климишин Данило   ·   ШІ-31

Ваші вісім стовпців:
   1. Телефон
   2. Примітка
   3. ступінь КТ
   4. Дата розтину
   5. Ліжко днів
   6. Гемогл1
   7. АЛАТ1
   8. БР заг1

Ваша пара для аналізу зв’язку: Гемогл1 та Гематокрит1

Ваше додаткове питання:
   Знайдіть у файлі стовпець, де одна й та сама величина записана у двох несумісних кодуваннях. Скільки спостережень втратить наївне обчислення середнього?


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Примітка,ступінь КТ,Дата розтину,Ліжко днів,Гемогл1,АЛАТ1,БР заг1
0,0958469788 дружина Ганна,NaN,сердній,2022-03-11,14,157,52.4,15.3
1,"0671778486, 0994887663 дружина Оксана","отримував актемра, часті носові кровотечі, втрата нюху",середній,NaT,22,145,22.0,61.8
2,"0958768045, чоловік Степан 0951784726",часта втрата свідомосі в стаціонарі,критичний,NaT,20,137,21.9,11.7
3,0959819331. дочка Валентина 0960845205,панкреонекроз,критичний,NaT,20,87,38.7,11.9
4,"0952294866, дочка Дарина 0678360161",NaN,NaN,NaT,27,126,97.7,13.6
5,0959922983 сестра Софія,NaN,середній,2021-12-26,9(5 у реан),134,112.0,11.9
6,"0998073702, дружина Наталія 0998382226","глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",важкий,NaT,18,151,60.4,15.3
7,"0637258771, син Сергій 0968413301",NaN,легкий,NaT,10,130,54.3,13.6


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Телефон       120 non-null    object        
 1   Примітка      45 non-null     str           
 2   ступінь КТ    42 non-null     str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   Ліжко днів    120 non-null    object        
 5   Гемогл1       119 non-null    object        
 6   АЛАТ1         115 non-null    float64       
 7   БР заг1       110 non-null    float64       
dtypes: datetime64[us](1), float64(2), object(3), str(2)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- Ліжко днів
    непорожніх 120 з 121 | унікальних 40 | типи: int×119, str×1
    приклади: ['14', '22'

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),

    'Телефон': ('ідентифікатор', 'Це унікальний номер кожного, тут немає сенсу шукати середнє і т.д'),
    'Примітка': ('Номінальна', 'Значення можна лише порівняти на рівність, порядку немає'),
    'ступінь КТ': ('Порядкова', 'Є природний порядок: легкий < середній < важкий < критичний. Але відстань між рівнями невідома.'),
    'Дата розтину': ('Інтервальна', 'Різниця дат має сенс, а нуль календаря умовний: "дата вдвічі більша" не означає нічого.'),
    'Ліжко днів': ('Відношень', 'Тривалість у днях має природний нуль, тому 20 днів справді вдвічі більше, ніж 10.'),
    'Гемогл1': ('Відношень', '	Концентрація гемоглобіну, де нуль означає відсутність речовини.'),
    'АЛАТ1': ('Відношень', 'Активність ферменту, де нуль означає відсутність активності'),
    'БР заг1': ('Відношень', 'Концентрація загального білірубіну, нуль природний')


}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,"Це унікальний номер кожного, тут немає сенсу шукати сере..."
1,Примітка,Номінальна,"Значення можна лише порівняти на рівність, порядку немає"
2,ступінь КТ,Порядкова,Є природний порядок: легкий < середній < важкий < критич...
3,Дата розтину,Інтервальна,"Різниця дат має сенс, а нуль календаря умовний: ""дата вд..."
4,Ліжко днів,Відношень,"Тривалість у днях має природний нуль, тому 20 днів справ..."
5,Гемогл1,Відношень,"\tКонцентрація гемоглобіну, де нуль означає відсутність ..."
6,АЛАТ1,Відношень,"Активність ферменту, де нуль означає відсутність активності"
7,БР заг1,Відношень,"Концентрація загального білірубіну, нуль природний"


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
ПОРЯДОК_КТ = ['легкий', 'середній', 'важкий', 'критичний']
ВИПРАВЛЕННЯ_КТ = {'сердній': 'середній'} 

def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    p = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    # 'Телефон': 'ідентифікатор'
    # 'Примітка': 'Номінальна',
    # 'ступінь КТ': 'Порядкова', 
    # 'Дата розтину': 'Інтервальна'
    # 'Ліжко днів': 'Відношень',
    # 'Гемогл1': 'Відношень', '
    # 'АЛАТ1': 'Відношень', 
    # 'БР заг1': 'Відношень'

    if ш in ('ідентифікатор',):
        return p
    

    if ш == 'порядкова':
        з = з.astype(str).str.strip().str.lower().replace(ВИПРАВЛЕННЯ_КТ)


    if ш in ('відношень', 'абсолютна'):
        до = len(з)
        з = pd.to_numeric(з, errors='coerce').dropna()
        p['втрачено при перетворенні'] = до - len(з)


    p['мода'] = з.mode().iloc[0]
    p['частоти'] = з.value_counts().head(5).to_dict()


    if ш == 'номінальна':
        return p

    if ш == 'порядкова':
        коди = з.map({рівень: i for i, рівень in enumerate(ПОРЯДОК_КТ)})
        p['не розпізнано'] = int(коди.isna().sum())
        коди = коди.dropna()
        q = коди.quantile([0.25, 0.5, 0.75], interpolation='lower')
        p['Q1'], p['медіана'], p['Q3'] = [ПОРЯДОК_КТ[int(k)] for k in q]
        return p

    p['Q1'] = з.quantile(0.25)
    p['медіана'] = з.median()
    p['Q3'] = з.quantile(0.75)
    p['середнє'] = з.mean()
    p['стандартне відхилення'] = з.std()

    if ш == 'інтервальна':
        return p

    p['коефіцієнт варіації'] = з.std() / з.mean()
    return p

    # ваш код: додайте моду й частоти для всіх шкал
    # для порядкової — медіану та квантилі
    # для інтервальної та вище — середнє й стандартне відхилення
    # для відношень і абсолютної — коефіцієнт варіації


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- Примітка  (Номінальна)
    {'непорожніх': 45, 'унікальних': 44, 'мода': 'Поліалергетик', 'частоти': {'Поліалергетик': 2, 'отримував актемра, часті носові кровотечі, втрата нюху': 1, 'часта втрата свідомосі в стаціонарі': 1, 'панкреонекроз': 1, 'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс': 1}}

--- ступінь КТ  (Порядкова)
    {'непорожніх': 42, 'унікальних': 12, 'мода': 'середній', 'частоти': {'середній': 11, 'важкий': 10, 'легкий': 6, 'серед ступ': 6, 'вкрай важкий': 4}, 'не розпізнано': 12, 'Q1': 'середній', 'медіана': 'середній', 'Q3': 'важкий'}

--- Дата розтину  (Інтервальна)
    {'непорожніх': 61, 'унікальних': 50, 'мода': Timestamp('2022-06-07 00:00:00'), 'частоти': {Timestamp('2022-06-07 00:00:00'): 3, Timestamp('2022-04-07 00:00:00'): 2, Timestamp('2022-04-27 00:00:00'): 2, Timestamp('2022-04-13 00:00:00'): 2, Timestamp('2022-03-30 00:00:00'): 2}, 'Q1': Timestamp('2022-03-29 00:00:00'), 'м

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [18]:
df[МОЇ].describe(include='all').T



,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Гемогл1,119.0,64.0,134.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
АЛАТ1,115.0,NaN,NaN,NaN,40.643826,0.6,13.0,37.4,60.3,138.0,32.210484
БР заг1,110.0,NaN,NaN,NaN,17.295273,7.43,13.6,15.3,18.3,84.0,10.486351


In [21]:
тел = pd.to_numeric(df['Телефон'], errors='coerce')
тел_середнє = тел.mean()

кт_унікальних = df['ступінь КТ'].nunique()

дати = df['Дата розтину'].dropna()
від_1970 = (дати - pd.Timestamp('1970-01-01')).dt.days
cv_1970 = від_1970.std() / від_1970.mean()

кт_унікальних = df['ступінь КТ'].nunique()

БЕЗГЛУЗДІ = [
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),

    ('Телефон', 'середнє', round(тел_середнє),
     'телефон є ідентифікатором: середнє номерів не відповідає жодній людині, '
     'а ще й рахується лише з 23 номерів, які Excel зробив числами, '
     'загубивши провідний нуль'),

    ('ступінь КТ', 'кількість унікальних (describe)', кт_унікальних,
     'шкала має чотири рівні, а describe рахує 12, бо кожен описок '
     'на кшталт "сердній" вважає окремою категорією; '
     'число відображає якість введення, а не різноманітність стану легень'),

    ('Дата розтину', 'коефіцієнт варіації', round(cv_1970, 4),
     'шкала інтервальна, нуль календаря умовний; CV ділить на середнє, '
     'тому залежить від початку відліку: від 1970 і від 2021 '
     'виходять зовсім різні значення для тих самих дат'),
    
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,середнє,7.587990e+08,телефон є ідентифікатором: середнє номерів не відповідає...
1,ступінь КТ,кількість унікальних (describe),1.200000e+01,"шкала має чотири рівні, а describe рахує 12, бо кожен оп..."
2,Дата розтину,коефіцієнт варіації,3.800000e-03,"шкала інтервальна, нуль календаря умовний; CV ділить на ..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [22]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Гемогл1 × Гематокрит1

Гемогл1: шкала — Відношень
Гематокрит1: шкала — не у вашому списку

  Гемогл1  Гематокрит1
0     157         48.0
1     145         42.2
2     137         40.0
3      87         22.0
4     126         40.0
5     134         39.0


In [36]:
from scipy import stats

# 'Ліжко днів': 'Відношень',
# 'Гемогл1': 'Відношень', '
# 'АЛАТ1': 'Відношень', 
# 'БР заг1': 'Відношень'

ОБРАНА_МІРА = 'кореляція Пірсона'        # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = 'гемоглобін і гематокрит мають шкалу відношень з природним нулем, тому застосовна кореляція Пірсона'    # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

гем = pd.to_numeric(df[A].astype(str).str.replace(',', '.').str.strip(), errors='coerce')
гкт = pd.to_numeric(df[B].astype(str).str.replace(',', '.').str.strip(), errors='coerce')

пара = pd.concat([гем, гкт], axis=1, keys=[A, B]).dropna()

r, p = stats.pearsonr(пара[A], пара[B])
print(f'Пірсон:  r = {r:.3f}, p = {p:.2g}, n = {len(пара)}')

# ваш код: обчисліть обрану міру


Пірсон:  r = 0.781, p = 1.3e-24, n = 114


In [38]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.




кт = (df['ступінь КТ'].astype(str).str.strip().str.lower()
      .replace(ВИПРАВЛЕННЯ_КТ))

коди_А = кт.map({'легкий': 0, 'середній': 1, 'важкий': 2, 'критичний': 3})
коди_Б = кт.map({'легкий': 1, 'середній': 2, 'важкий': 5, 'критичний': 10})

д = pd.concat([коди_А, коди_Б, гем], axis=1, keys=['А', 'Б', 'гем']).dropna()

r_А, _ = stats.pearsonr(д['А'], д['гем'])
r_Б, _ = stats.pearsonr(д['Б'], д['гем'])

print(f'Пацієнтів з обома значеннями: {len(д)}')
print()
print(f'Пірсон,  коди 0-1-2-3:  r   = {r_А:+.3f}')
print(f'Пірсон,  коди 1-2-5-10: r   = {r_Б:+.3f}')
print()

ПОЯСНЕННЯ_КОНТРПРИКЛАДУ = (
    'Ступінь КТ має порядкову шкалу: відомо лише, що легкий < середній < важкий < критичний, '
    'а відстані між рівнями не визначені. Тому числові коди рівнів обираються довільно, '
    'і будь-яке зростаюче кодування однаково правильне. Пірсон працює з відстанями між '
    'значеннями, тож при двох рівноправних кодуваннях дає різні коефіцієнти: число '
    'описує обране кодування, а не зв\'язок ступеня КТ з гемоглобіном.'
)
print(ПОЯСНЕННЯ_КОНТРПРИКЛАДУ)


Пацієнтів з обома значеннями: 30

Пірсон,  коди 0-1-2-3:  r   = -0.278
Пірсон,  коди 1-2-5-10: r   = -0.394

Ступінь КТ має порядкову шкалу: відомо лише, що легкий < середній < важкий < критичний, а відстані між рівнями не визначені. Тому числові коди рівнів обираються довільно, і будь-яке зростаюче кодування однаково правильне. Пірсон працює з відстанями між значеннями, тож при двох рівноправних кодуваннях дає різні коефіцієнти: число описує обране кодування, а не зв'язок ступеня КТ з гемоглобіном.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [39]:
ПІДГОТОВКА = {
    # 'стовпець': 'спосіб і коротке обґрунтування',

    'Телефон':      'не беремо в модель: ідентифікатор, не несе інформації про стан пацієнта',
    'Примітка':     'не беремо напряму: майже всі тексти унікальнi',
    'ступінь КТ':   'цілі коди 0-3 у заданому порядку легкий < середній < важкий < критичний',
    'Дата розтину': 'бінарна ознака летального наслідку + тривалість у днях від опорної дати',
    'Ліжко днів':   'витягти число з рядків, потім стандартизувати',
    'Гемогл1':      'звести до г/л, стандартизувати ',
    'АЛАТ1':        'логарифмувати через правий хвіст, потім стандартизувати',
    'БР заг1':      'логарифмувати через правий хвіст, потім стандартизувати',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,"не беремо в модель: ідентифікатор, не несе інформації пр..."
1,Примітка,Номінальна,не беремо напряму: майже всі тексти унікальнi
2,ступінь КТ,Порядкова,цілі коди 0-3 у заданому порядку легкий < середній < важ...
3,Дата розтину,Інтервальна,бінарна ознака летального наслідку + тривалість у днях в...
4,Ліжко днів,Відношень,"витягти число з рядків, потім стандартизувати"
5,Гемогл1,Відношень,"звести до г/л, стандартизувати"
6,АЛАТ1,Відношень,"логарифмувати через правий хвіст, потім стандартизувати"
7,БР заг1,Відношень,"логарифмувати через правий хвіст, потім стандартизувати"


In [40]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал


підготовлено = pd.DataFrame(index=df.index)

# --- 1. Порядкова: ступінь КТ -> коди із заданим порядком ---
кт = (df['ступінь КТ'].astype(str).str.strip().str.lower()
      .replace(ВИПРАВЛЕННЯ_КТ))
підготовлено['КТ_код'] = кт.map({р: i for i, р in enumerate(ПОРЯДОК_КТ)})

# --- 2. Інтервальна: Дата розтину -> бінарна ознака + тривалість ---
дата = df['Дата розтину']
підготовлено['летальний'] = дата.notna().astype(int)
опорна = дата.min()
підготовлено['днів_від_першого_розтину'] = (дата - опорна).dt.days

# --- 3. Відношень: АЛАТ1 -> логарифм + стандартизація ---
алат_лог = np.log(df['АЛАТ1'])
підготовлено['АЛАТ1_лог_ст'] = (алат_лог - алат_лог.mean()) / алат_лог.std()

# --- 4. Відношень: Ліжко днів -> чистка + стандартизація ---
дні = df['Ліжко днів'].astype(str).str.extract(r'^\s*(\d+)')[0].astype(float)
підготовлено['Ліжко_днів_ст'] = (дні - дні.mean()) / дні.std()

# --- Ідентифікатор: Телефон у модель не потрапляє ---

print(підготовлено.describe().round(2).T)
підготовлено.head(8)


                          count    mean    std   min     25%     50%     75%  \
КТ_код                     30.0    1.33   0.92  0.00    1.00    1.00    2.00   
летальний                 121.0    0.50   0.50  0.00    0.00    1.00    1.00   
днів_від_першого_розтину   61.0  455.56  72.57  0.00  447.00  472.00  486.00   
АЛАТ1_лог_ст              115.0    0.00   1.00 -2.22   -0.28    0.39    0.69   
Ліжко_днів_ст             120.0    0.00   1.00 -1.31   -0.61   -0.22    0.32   

                             max  
КТ_код                      3.00  
летальний                   1.00  
днів_від_першого_розтину  532.00  
АЛАТ1_лог_ст                1.21  
Ліжко_днів_ст               4.72  


,КТ_код,летальний,днів_від_першого_розтину,АЛАТ1_лог_ст,Ліжко_днів_ст
0,1.0,1,429.0,0.599440,-0.300174
1,1.0,0,NaN,0.052662,0.318210
2,3.0,0,NaN,0.049792,0.163614
3,3.0,0,NaN,0.408499,0.163614
4,NaN,0,NaN,0.991943,0.704700
5,1.0,1,354.0,1.078003,-0.686664
6,2.0,0,NaN,0.688956,0.009018
7,0.0,0,NaN,0.621880,-0.609366


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [41]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [42]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [ ]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [ ]:
# ваш код: три правильні обчислення замість хибних


---
# Етап 7. Висновки та індивідуальне питання

In [ ]:
ВИСНОВКИ = [
    # три висновки, кожен — повне речення про ваші дані, а не загальна фраза
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

In [ ]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = ''
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [ ]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        '',
    'етап 2 — допустимі статистики':   '',
    'етап 3 — безглузді числа':        '',
    'етап 4 — міра зв’язку':           '',
    'етап 5 — підготовка до моделі':   '',
    'етап 6 — виправлення аналізу':    '',
    'етап 7 — висновки й відповідь':   '',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = ''
ПІДТВЕРДЖУЮ = False     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [ ]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.